# Librerías

In [ ]:
import cv2
import argparse
import os
import imageio
import kornia
import torch
import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
import urllib.request as request;
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from torch import nn
from tqdm import tqdm
from deepface import DeepFace
from ipywidgets import interact, FloatSlider
from skimage import data, img_as_float
from skimage.metrics import structural_similarity as ssim



# 3. Funciones generales útiles

In [ ]:
def plot_histogram_rgb(image, vis = False):
    '''
    Función que permite graficar los histogramas de las componentes RGB de una imagen.

    Args:
        image: imagen a procesar.
    '''
    # Cálculo de los histogramas
    hist_r = cv2.calcHist([image], [0], None, [256], [0, 256])
    hist_g = cv2.calcHist([image], [1], None, [256], [0, 256])
    hist_b = cv2.calcHist([image], [2], None, [256], [0, 256])

    # Graficar cada uno de los histogramas
    if vis:
        fig, ax = plt.subplots(1, 3, figsize=(30, 10))
        ax[0].plot(hist_r.flatten(), color='red')
        ax[0].set_title('Histograma Rojo')
        ax[0].set_xlabel('Intensidad de iluminación')
        ax[0].set_ylabel('Cantidad de pixeles')
        ax[0].set_xlim([0, 256])

        ax[1].plot(hist_g.flatten(), color='green')
        ax[1].set_title('Histograma Verde')
        ax[1].set_xlabel('Intensidad de iluminación')
        ax[1].set_ylabel('Cantidad de pixeles')
        ax[1].set_xlim([0, 256])

        ax[2].plot(hist_b.flatten(), color='blue')
        ax[2].set_title('Histograma Azul')
        ax[2].set_xlabel('Intensidad de iluminación')
        ax[2].set_ylabel('Cantidad de pixeles')
        ax[2].set_xlim([0, 256])

        plt.show()

    return hist_r, hist_g, hist_b

def saturated_histogram(array_image):
    '''
    Función que permite saturar los valores de un histograma, siendo la distribución:
    - 0 si el valor es menor a 0.
    - 255 si el valor es mayor a 255.

    Args:
        array_image: arreglo de la imagen a procesar.

    Returns:
        array_image: arreglo de la imagen con los valores saturados.
    '''
    array_image = array_image.astype(np.uint8)

    # Saturación de los valores
    array_image[array_image < 0] = 0
    array_image[array_image > 255] = 255

    return array_image

# 4. Mejora usando Modelos Clásicos

In [ ]:
def contrast_extend(image, channel, lim_a, lim_b):
    '''
    Función que permite extender el contraste de una imagen.

    Args:
        image: imagen a procesar.
        channel: canal de la imagen a procesar.
        lim_a: límite inferior.
        lim_b: límite superior.

    Returns:
        image: imagen con el contraste extendido.
    '''
    # Estiramiento lineal de contraste
    image[:, :, channel] = np.clip((image[:, :, channel] - lim_a) * (255 / (lim_b - lim_a)), 0, 255)

    # Asegurarse de que el tipo de dato sea uint8
    image[:, :, channel] = image[:, :, channel].astype(np.uint8)

    return image

def equal_hist(image):
    '''
    Función que se encarga de la ecualización del histograma
    de una imagen de entrada, aplicando una look-up table.

    Args:
        image: imagen a procesar.

    Returns:
        image_eq: imagen con el histograma ecualizado.
    '''
    # Creación de una copia para preservar la original
    image_eq = image.copy()
    L = 256

    # Ecualización de cada canal por separado
    for channel in range(3):
        # Cálculo del histograma y su acumulado para el canal
        hist = cv2.calcHist([image], [channel], None, [L], [0, L])
        cdf = hist.cumsum()  # Función de distribución acumulativa

        # Normalización de cdf
        cdf_max = cdf.max()
        cdf_min = cdf.min()
        cdf_normalized = (cdf - cdf_min) * (L - 1) / (cdf_max- cdf_min)
        cdf_normalized = cdf_normalized.astype('uint8')  # Conversión a enteros

        # Mapear los valores antiguos a los nuevos usando la look-up table
        image_eq[:, :, channel] = cdf_normalized[image[:, :, channel]]

    return image_eq

def clahe_enhancement(image, clahe_rgb = False):
    '''
    Función que permite realizar la mejora de contraste de una imagen
    mediante CLAHE.

    Args:
        image: imagen a procesar cargada en RGB.
        clahe_rgb: booleano que indica si se aplica CLAHE en espacio rgb
        (todos los canales) o en el espacio HSV (solo el canal Value).

    Returns:
        image_clahe: imagen con el contraste mejorado.
    '''
    # Crear una copia de la imagen sobre la cual trabajar
    image_clahe = image.copy()

    # Crear estructura CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    if clahe_rgb:
        # Aplicar CLAHE a cada canal por separado
        for channel in range(3):
            image_clahe[:, :, channel] = clahe.apply(image_clahe[:, :, channel])

    else:
        # Transformar imagen al dominio HSV
        image_clahe = cv2.cvtColor(image_clahe, cv2.COLOR_RGB2HSV)

        # Aplicar CLAHE al tercer canal (canal Value)
        image_clahe[:, :, 2] = clahe.apply(image_clahe[:, :, 2])

        # Transformar imagen de vuelta al dominio RGB
        image_clahe = cv2.cvtColor(image_clahe, cv2.COLOR_HSV2RGB)

    image_clahe = image_clahe.astype(np.uint8)
    return image_clahe

# 5. Mejora usando *Bread*

In [ ]:
# Load some scripts from remote.
exec(request.urlopen('https://github.com/mingcv/Bread_Colab/raw/main/colab_utils.py').read(), globals())
exec(request.urlopen(locate_resource('networks.py')).read(), globals())

## Clase del modelo

In [ ]:
# Defination of the Bread Framework.
class ModelBreadNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.eps = 1e-6
        self.model_ianet = IAN(in_channels=1, out_channels=1)
        self.model_nsnet = ANSN(in_channels=2, out_channels=1)
        self.model_canet = FuseNet(in_channels=4, out_channels=2)

        self.load_weight(self.model_ianet, './IANet_335.pth')
        self.load_weight(self.model_nsnet, './NSNet_422.pth')
        self.load_weight(self.model_canet, './FuseNet_CA_MEF_251.pth')

    def load_weight(self, model, weight_pth):
        if model is not None:
            state_dict = torch.load(weight_pth)
            ret = model.load_state_dict(state_dict, strict=True)
            print(ret)

    def noise_syn_exp(self, illumi, strength):
        return torch.exp(-illumi) * strength

    def forward(self, image, gamma=1., strength=0.1):
        # Color space mapping
        texture_in, cb_in, cr_in = torch.split(kornia.color.rgb_to_ycbcr(image), 1, dim=1)

        # Illumination prediction
        texture_in_down = F.interpolate(texture_in, scale_factor=0.5, mode='bicubic', align_corners=True)
        texture_illumi = self.model_ianet(texture_in_down)
        texture_illumi = F.interpolate(texture_illumi, scale_factor=2, mode='bicubic', align_corners=True)

        # Illumination adjustment
        texture_illumi = torch.clamp(texture_illumi ** gamma, 0., 1.)
        texture_ia = texture_in / torch.clamp_min(texture_illumi, self.eps)
        texture_ia = torch.clamp(texture_ia, 0., 1.)

        # Noise suppression and fusion
        attention = self.noise_syn_exp(texture_illumi, strength)
        texture_res = self.model_nsnet(torch.cat([texture_ia, attention], dim=1))
        texture_ns = texture_ia + texture_res

        # Further preserve the texture under brighter illumination
        texture_ns = texture_illumi * texture_in + (1 - texture_illumi) * texture_ns
        texture_ns = torch.clamp(texture_ns, 0, 1)

        # Color adaption
        colors = self.model_canet(
            torch.cat([texture_in, cb_in, cr_in, texture_ns], dim=1))
        cb_out, cr_out = torch.split(colors, 1, dim=1)
        cb_out = torch.clamp(cb_out, 0, 1)
        cr_out = torch.clamp(cr_out, 0, 1)

        # Color space mapping
        image_out = kornia.color.ycbcr_to_rgb(
            torch.cat([texture_ns, cb_out, cr_out], dim=1))

        # Further preserve the color under brighter illumination
        img_fusion = texture_illumi * image + (1 - texture_illumi) * image_out
        _, cb_fuse, cr_fuse = torch.split(kornia.color.rgb_to_ycbcr(img_fusion), 1, dim=1)
        image_out = kornia.color.ycbcr_to_rgb(
            torch.cat([texture_ns, cb_fuse, cr_fuse], dim=1))
        image_out = torch.clamp(image_out, 0, 1)

        # outputs: texture_ia, texture_ns, image_out, texture_illumi, texture_res
        return image_out

model = ModelBreadNet().eval().cuda()

## Ejemplo de notebook

In [ ]:
# Load image from a url and convert it into pytorch tensor
im = imageio.imread(locate_resource("images/Balloons.png"))
im = size_round(im)
imshow(im)

im_in = numpy_to_tensor(im).cuda()

In [ ]:
def filtering(gamma=1.0, strength=0.1):  
    im_adj = model(im_in, gamma, strength)
    im_adj = tensor_to_numpy(im_adj)
    imshow(im_adj)


interact(filtering, 
     gamma=FloatSlider(min=0., max=1.5, step=0.1, value=1.0, continuous_update=True),    # gamma correction for the resulting illumination
     strength=FloatSlider(min=0., max=0.2, step=0.01, value=0.05, continuous_update=True)); # controlling the denoising strength

# 6. Evaluación y Comparación de Métodos Implementados

In [ ]:
def mse_calc(image1, image2):
    '''
    Calcula el error cuadrático medio (MSE) entre dos imágenes.

    Args:
        image1: primera imagen a comparar.
        image2: segunda imagen a comparar.

    Returns:
        mse: valor del error cuadrático medio.
    '''
    # Asegurarse de que las imágenes tengan las mismas dimensiones
    if image1.shape != image2.shape:
        raise ValueError("Las imágenes deben tener las mismas dimensiones")

    # Calcular el MSE
    mse = np.mean((image1.astype("float") - image2.astype("float")) ** 2)
    return mse

def ssim_calc(image1, image2):
    '''
    Calcula el Índice de Similitud Estructural (SSIM) entre dos imágenes.

    Args:
        image1: primera imagen a comparar.
        image2: segunda imagen a comparar.

    Returns:
        ssim_value: valor del índice de similitud estructural.
    '''
    # Asegurarse de que las imágenes tengan las mismas dimensiones
    if image1.shape != image2.shape:
        raise ValueError("Las imágenes deben tener las mismas dimensiones")

    # Calcular el SSIM
    ssim_value, _ = ssim(image1, image2, full=True, multichannel=True)
    return ssim_value

# def compare_images(imageA, imageB, title):
# 	# compute the mean squared error and structural similarity
# 	# index for the images
# 	m = mse_calc(imageA, imageB)
# 	s = ssim_calc(imageA, imageB)
# 	# setup the figure
# 	fig = plt.figure(title)
# 	plt.suptitle("MSE: %.2f, SSIM: %.2f" % (m, s))
# 	# show first image
# 	ax = fig.add_subplot(1, 2, 1)
# 	plt.imshow(imageA, cmap = plt.cm.gray)
# 	plt.axis("off")
# 	# show the second image
# 	ax = fig.add_subplot(1, 2, 2)
# 	plt.imshow(imageB, cmap = plt.cm.gray)
# 	plt.axis("off")
# 	# show the images
# 	plt.show()

# 7. Aplicación Real

In [ ]:
def visualize_detection(img_path):
  """
  Visualizes the image with predicted bounding boxes from DeepFace.

  Args:
    img_path: Path to the image.
    dfs: The results from DeepFace.find.
  """

  dfs = DeepFace.find(
    img_path = img_path,
    db_path = "/content/",
    enforce_detection=False
    )

  img = cv2.imread(img_path)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

  for df in dfs:
    # if empty dataframe
    if df.empty:
      continue
    # if not empty dataframe
    target_x, target_y, target_w, target_h = df['target_x'], df['target_y'], df['target_w'], df['target_h']
    #source_x, source_y, source_w, source_h = df['source_x'], df['source_y'], df['source_w'], df['source_h']
    cv2.rectangle(img, (int(target_x), int(target_y)), (int(target_x + target_w), int(target_y + target_h)), (0, 255, 0), 2)
    #cv2.rectangle(img, (int(source_x), int(source_y)), (int(source_x + source_w), int(source_y + source_h)), (255, 0, 0), 2)

  plt.imshow(img)
  plt.axis('off')
  plt.show()

visualize_detection("/content/1.png")